# IoT Cyber-Physical Security Platform
## Quick Start Notebook

This notebook demonstrates the full pipeline: data generation, model training, evaluation, and visualization.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config.settings import RANDOM_SEED, DATA_ROOT, MODEL_ROOT
from src.preprocessing.physical import PhysicalPreprocessor
from src.preprocessing.cyber import CyberPreprocessor
from src.models.isolation_forest import IsolationForestModel
from src.models.autoencoder import AutoencoderModel
from src.models.random_forest import RandomForestModel
from src.models.xgboost import XGBoostModel
from src.decision.engine import DecisionEngine

print('All imports successful!')

## 1. Generate Synthetic Data

In [ ]:
rng = np.random.RandomState(RANDOM_SEED)
n = 5000

# Physical sensor data
labels_phys = np.zeros(n, dtype=int)
labels_phys[1000:1100] = 1
labels_phys[3500:3550] = 1

df_physical = pd.DataFrame({
    'FIT101': rng.normal(2.5, 0.3, n),
    'LIT101': rng.normal(500, 20, n),
    'FIT102': rng.normal(2.4, 0.3, n),
    'LIT102': rng.normal(300, 15, n),
    'FIT103': rng.normal(1.2, 0.2, n),
    'LIT103': rng.normal(600, 25, n),
    'attack_label': labels_phys,
})

# Cyber network data
cyber_labels = rng.choice(['Benign', 'DDoS', 'DoS', 'BruteForce'], n, p=[0.5, 0.2, 0.2, 0.1])
df_cyber = pd.DataFrame({
    'flow_duration': np.abs(rng.exponential(5, n)),
    'packet_count': rng.randint(1, 10000, n),
    'byte_count': rng.randint(100, 1000000, n),
    'packet_rate': np.abs(rng.exponential(50, n)),
    'byte_rate': np.abs(rng.exponential(5000, n)),
    'attack_label': cyber_labels,
})

print(f'Physical: {df_physical.shape}, Cyber: {df_cyber.shape}')
print(f'Attack rate (physical): {labels_phys.mean():.2%}')
print(f'Attack distribution (cyber): {pd.Series(cyber_labels).value_counts().to_dict()}')

## 2. Train Physical Anomaly Models

In [ ]:
phys_features = [c for c in df_physical.columns if c != 'attack_label']
X_phys = df_physical[phys_features].values
y_phys = df_physical['attack_label'].values

from sklearn.model_selection import train_test_split
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    X_phys, y_phys, test_size=0.2, random_state=42, stratify=y_phys
)

preprocessor_p = PhysicalPreprocessor()
Xp_train_scaled = preprocessor_p.fit_transform(pd.DataFrame(Xp_train, columns=phys_features), phys_features)
Xp_test_scaled = preprocessor_p.transform(pd.DataFrame(Xp_test, columns=phys_features))

# Isolation Forest
if_model = IsolationForestModel(n_estimators=100, contamination=0.1, random_state=42)
if_model.fit(Xp_train_scaled, phys_features)
if_metrics = if_model.evaluate(Xp_test_scaled, yp_test)
print(f'Isolation Forest - F1: {if_metrics["f1"]:.4f}, ROC-AUC: {if_metrics["roc_auc"]:.4f}')

# Autoencoder
ae_model = AutoencoderModel(input_dim=len(phys_features), encoding_dim=8,
                             hidden_dims=[16], epochs=10, batch_size=64)
ae_model.fit(Xp_train_scaled, phys_features)
ae_metrics = ae_model.evaluate(Xp_test_scaled, yp_test)
print(f'Autoencoder - F1: {ae_metrics["f1"]:.4f}, ROC-AUC: {ae_metrics["roc_auc"]:.4f}')

## 3. Train Cyber Threat Models

In [ ]:
cyber_features = [c for c in df_cyber.columns if c != 'attack_label']

preprocessor_c = CyberPreprocessor()
X_cyber_scaled, y_cyber_encoded = preprocessor_c.fit_transform(
    df_cyber, cyber_features, 'attack_label'
)
label_classes = preprocessor_c.label_classes

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cyber_scaled, y_cyber_encoded, test_size=0.2, random_state=42
)

# Random Forest
rf_model = RandomForestModel(n_estimators=100, random_state=42)
rf_model.fit(Xc_train, yc_train, feature_names=cyber_features, label_classes=label_classes)
rf_metrics = rf_model.evaluate(Xc_test, yc_test)
print(f'Random Forest - Accuracy: {rf_metrics["accuracy"]:.4f}, F1: {rf_metrics["f1"]:.4f}')

# XGBoost
xgb_model = XGBoostModel(n_estimators=100, random_state=42)
xgb_model.fit(Xc_train, yc_train, feature_names=cyber_features, label_classes=label_classes)
xgb_metrics = xgb_model.evaluate(Xc_test, yc_test)
print(f'XGBoost - Accuracy: {xgb_metrics["accuracy"]:.4f}, F1: {xgb_metrics["f1"]:.4f}')

## 4. Decision Engine - Unified Risk

In [ ]:
engine = DecisionEngine()

# Test with various score combinations
results = []
for phys_score in np.linspace(0, 1, 11):
    for cyber_score in np.linspace(0, 1, 11):
        result = engine.compute_risk(
            physical_score=float(phys_score),
            cyber_score=float(cyber_score),
        )
        results.append({
            'Physical Score': phys_score,
            'Cyber Score': cyber_score,
            'Unified Score': result.unified_risk_score,
            'Risk Level': str(result.risk_level),
        })

df_results = pd.DataFrame(results)
pivot = df_results.pivot(index='Physical Score', columns='Cyber Score', values='Unified Score')

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r')
plt.title('Unified Risk Score Matrix')
plt.ylabel('Physical Anomaly Score')
plt.xlabel('Cyber Threat Score')
plt.tight_layout()
plt.show()

## 5. Model Comparison Visualization

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Isolation Forest', 'Autoencoder', 'Random Forest', 'XGBoost'],
    'F1 Score': [
        if_metrics['f1'], ae_metrics['f1'],
        rf_metrics['f1'], xgb_metrics['f1'],
    ],
    'ROC-AUC': [
        if_metrics['roc_auc'], ae_metrics['roc_auc'],
        rf_metrics.get('roc_auc', 0), xgb_metrics.get('roc_auc', 0),
    ],
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
axes[0].barh(comparison['Model'], comparison['F1 Score'], color=colors)
axes[0].set_xlabel('F1 Score')
axes[0].set_title('F1 Score Comparison')
axes[0].set_xlim(0, 1)

axes[1].barh(comparison['Model'], comparison['ROC-AUC'], color=colors)
axes[1].set_xlabel('ROC-AUC')
axes[1].set_title('ROC-AUC Comparison')
axes[1].set_xlim(0, 1)

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(comparison.to_string(index=False))